# Lecture 7.5: Orchestrating Multi-Agent Pipelines

**Section 07: Subagents: Delegation & Multi-Agent Workflows**

The Bike Repair Co-op has three specialists: a parts clerk, a triage mechanic and a shift planner. In this notebook you combine them into pipelines, using nothing but goal prompts and agent descriptions.

| Pattern | When to use it | What you check |
|---|---|---|
| Parallel | Jobs that don't need each other's results | A timeline of when each specialist started and finished |
| Sequential | A later job needs an earlier job's output | The prompt the main agent wrote for the next specialist |
| Mixed pipeline | Both at once, then one combined result | A briefing file on disk |
| Keeping output exact | You need a specialist's words untouched | The briefing compared line by line with the specialist's report |
| Resuming a specialist | A follow-up question for the same specialist | The specialist's agent ID and the follow-up call |

There is no orchestration code in this notebook. Every workflow comes from the prompt you write, and the main agent decides how to carry it out. Agent output varies from run to run, so each check looks for facts (order IDs, part names, files on disk) rather than exact sentences.

## Cell 1: Install the Claude Agent SDK

This cell installs the Claude Agent SDK, the Python package that runs the agent loop and the specialists you meet in this notebook. The package bundles the Claude Code CLI that does the actual work, so nothing else needs installing.

The version is pinned so that every run behaves like the one in the video. The comment in the cell explains how to install the latest release instead. If the package is already installed in this Colab session, pip notices and finishes quickly.

In [ ]:
# We pin the Claude Agent SDK to a specific version so every example runs
# exactly as shown in the course. To use the latest version, run:
# pip install -U claude-agent-sdk
# Or substitute your preferred version below. Newer versions may behave
# differently from the videos.
!pip install claude-agent-sdk==0.2.174 -q

## Cell 2: Set your API key from Colab Secrets

The SDK reads your Anthropic API key from the `ANTHROPIC_API_KEY` environment variable. In Colab, the safe place to keep the key is Colab Secrets, so it never appears in the notebook.

If you haven't added the secret yet:

1. Click the **key icon** in Colab's left sidebar.
2. Click **Add new secret**.
3. Name it exactly `ANTHROPIC_API_KEY`.
4. Paste your key into the value field. You create keys in the Claude Console at https://platform.claude.com/ under **API Keys**.
5. Switch **Notebook access** on for this notebook.

The cell reads the secret and copies it into the environment, where the SDK looks for it.

> **Running locally instead of Colab?** Set the variable in your terminal before starting Jupyter, for example `export ANTHROPIC_API_KEY=...`, and skip this cell.

In [ ]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Cell 3: Choose the model

`MODEL_NAME` is the one place this notebook names a model. The main agent and all three specialists use it, so changing this variable changes the model across the entire notebook.

The course default is Claude Haiku 5.5. To try a different model, check the models page linked in the cell for current names.

In [ ]:
# See latest models at: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-5-5"

## Cell 4: Create the co-op files

This cell builds the co-op's paperwork under `/content/bikecoop/`. A fixed random seed means you get the same files every time you run it.

| File | Contents | Built-in facts |
|---|---|---|
| `work_orders/WO-101.txt` to `WO-140.txt` | 40 repair tickets: customer, bike, issue, skill needed, priority, safety flag | Exactly **five** are `Priority: urgent` with `Safety issue: yes`. No other order is urgent or flagged. |
| `inventory.csv` | 15 parts with `on_hand` and `reorder_level` | Exactly **four** parts have `on_hand` below `reorder_level`. |
| `volunteers.txt` | 6 volunteers with their skills and shift days | Every skill an urgent order needs is covered by at least one volunteer. |

Those built-in facts are the ground truth for the whole notebook. Later cells check the agents' work against them, which is far more reliable than judging the wording of a reply.

The cell also deletes any `briefing*.md` left over from an earlier run, so the file checks later on start from a clean folder.

In [ ]:
import csv
import random
from pathlib import Path

random.seed(75)

BASE = Path("/content/bikecoop")
ORDERS = BASE / "work_orders"
ORDERS.mkdir(parents=True, exist_ok=True)
for old in list(ORDERS.glob("*.txt")) + list(BASE.glob("briefing*.md")):
    old.unlink()

CUSTOMERS = ["Priya", "Tom", "Aiko", "Marcus", "Lena", "Omar", "Grace",
             "Ravi", "Sofia", "Kwame", "Hana", "Diego"]
BIKES = ["Commuter hybrid", "Road bike", "Mountain bike", "Cargo bike",
         "Folding bike", "Kids' bike"]

# The five urgent safety issues: (skill needed, issue)
URGENT_ISSUES = [
    ("brakes", "Rear brake cable frayed; lever pulls all the way to the bar"),
    ("wheels", "Front wheel quick release will not lock"),
    ("frame", "Crack spotted next to the seat tube weld"),
    ("brakes", "Hydraulic brake leaking fluid onto the front rotor"),
    ("drivetrain", "Chain snapped once on a hill and now skips under load"),
]
ROUTINE_ISSUES = [
    ("fit", "Saddle creaks when pedalling"),
    ("drivetrain", "Gears need re-indexing"),
    ("wheels", "Slow puncture in the rear tube"),
    ("accessories", "Bell missing"),
    ("electrical", "Rear light flickers"),
    ("fit", "Bar tape worn through"),
    ("accessories", "Kickstand loose"),
    ("wheels", "Spoke tension check"),
    ("brakes", "Brake pads squeal but stop fine"),
]

order_ids = [f"WO-{n}" for n in range(101, 141)]
urgent_ids = sorted(random.sample(order_ids, 5))
urgent_issue = dict(zip(urgent_ids, URGENT_ISSUES))

for i, oid in enumerate(order_ids):
    if oid in urgent_issue:
        skill, issue = urgent_issue[oid]
        priority, safety = "urgent", "yes"
    else:
        skill, issue = random.choice(ROUTINE_ISSUES)
        priority, safety = random.choice(["normal", "normal", "low"]), "no"
    (ORDERS / f"{oid}.txt").write_text(
        f"Order ID: {oid}\n"
        f"Received: 2026-11-{(i % 14) + 2:02d}\n"
        f"Customer: {random.choice(CUSTOMERS)}\n"
        f"Bike: {random.choice(BIKES)}\n"
        f"Issue: {issue}\n"
        f"Skill needed: {skill}\n"
        f"Priority: {priority}\n"
        f"Safety issue: {safety}\n"
    )

# 15 parts; exactly four are below their reorder level
PARTS = [
    ("brake cable", 2, 5), ("disc brake pads", 12, 6), ("rim brake pads", 9, 4),
    ("inner tube 700c", 3, 10), ("inner tube 26in", 14, 6), ("chain 9-speed", 1, 4),
    ("quick release skewer", 7, 3), ("hydraulic brake fluid", 1, 3), ("spokes", 60, 20),
    ("bar tape", 8, 4), ("saddle clamp", 6, 2), ("rear LED light", 10, 4),
    ("bell", 11, 5), ("tyre levers", 20, 8), ("cable housing", 15, 6),
]
with open(BASE / "inventory.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["part", "on_hand", "reorder_level"])
    writer.writerows(PARTS)

VOLUNTEERS = [
    ("Asha", "brakes, wheels", "Tue, Sat"),
    ("Ben", "drivetrain, fit", "Mon, Thu"),
    ("Carmen", "frame, brakes", "Wed, Sat"),
    ("Dev", "wheels, electrical", "Tue, Fri"),
    ("Elena", "drivetrain, frame", "Thu, Sun"),
    ("Farid", "accessories, fit", "Mon, Wed"),
]
(BASE / "volunteers.txt").write_text(
    "\n".join(f"{name} | skills: {skills} | shifts: {days}"
              for name, skills, days in VOLUNTEERS) + "\n"
)

print("Co-op files created under", BASE)

## Cell 5: Verify the files and print the ground truth

Before any agent runs, this cell shows you exactly what is on disk:

- the directory tree,
- one sample work order, so you know the ticket format,
- the full inventory and the volunteer list.

Then it reads the files back and computes the two answers every later cell checks against:

| Variable | How it is computed |
|---|---|
| `LOW_STOCK` | Parts whose `on_hand` is below `reorder_level` in `inventory.csv` |
| `URGENT_IDS` | Orders containing both `Priority: urgent` and `Safety issue: yes` |

It also lists which volunteers have the skill each urgent order needs, which confirms every urgent order can be staffed. These values come from the files, not from an agent, so they are the ground truth.

In [ ]:
print("Directory tree:")
for path in sorted(BASE.rglob("*")):
    depth = len(path.relative_to(BASE).parts) - 1
    print("   " * depth + path.name)

print("\nSample work order (WO-101.txt):")
print((ORDERS / "WO-101.txt").read_text())
print("inventory.csv:")
print((BASE / "inventory.csv").read_text())
print("volunteers.txt:")
print((BASE / "volunteers.txt").read_text())

# Ground truth, computed from the files themselves
with open(BASE / "inventory.csv") as f:
    LOW_STOCK = [row["part"] for row in csv.DictReader(f)
                 if int(row["on_hand"]) < int(row["reorder_level"])]

URGENT_IDS, NEEDS = [], {}
for path in sorted(ORDERS.glob("*.txt")):
    text = path.read_text()
    if "Priority: urgent" in text and "Safety issue: yes" in text:
        URGENT_IDS.append(path.stem)
        NEEDS[path.stem] = text.split("Skill needed: ")[1].splitlines()[0]

print("GROUND TRUTH")
print(f"Low-stock parts ({len(LOW_STOCK)}):", ", ".join(LOW_STOCK))
print(f"Urgent safety orders ({len(URGENT_IDS)}):", ", ".join(URGENT_IDS))
for oid in URGENT_IDS:
    able = [v[0] for v in VOLUNTEERS if NEEDS[oid] in v[1]]
    print(f"  {oid} needs {NEEDS[oid]:<10} -> can be staffed by {', '.join(able)}")

## Cell 6: Define the three specialists

Each specialist is an `AgentDefinition`. The main agent sees only each specialist's **description**, so the description is where you state its job and what it needs as input. That matters for pipelines: the `shift-planner` description says it needs order IDs in its prompt, which tells the main agent what to hand over.

| Field | `parts-clerk` | `triage-mechanic` | `shift-planner` |
|---|---|---|---|
| Job | Find parts below their reorder level | Rank the urgent safety orders | Assign volunteers to given orders |
| Input it needs | Nothing beyond `inventory.csv` | Nothing beyond the work orders folder | **The order IDs, in its prompt** |
| `tools` | `Read`, `Glob`, `Grep` | `Read`, `Glob`, `Grep` | `Read`, `Glob`, `Grep` |
| `model` | `MODEL_NAME` | `MODEL_NAME` | `MODEL_NAME` |

All three are read-only. None of them has `Write`, so a specialist cannot create a file even if a prompt asks it to. Only the main agent will write files, and only in the cells that allow it.

The `prompt` field is each specialist's own system prompt. Each one tells the specialist to keep its report short and not to create files.

> Field names in `AgentDefinition` follow the wire format, so multi-word fields such as `disallowedTools` stay in camelCase. The fields used here are all single words.

In [ ]:
from claude_agent_sdk import AgentDefinition

AGENTS = {
    "parts-clerk": AgentDefinition(
        description=(
            "Checks the co-op parts inventory and reports every part below its "
            "reorder level. Needs no input: it reads /content/bikecoop/inventory.csv."
        ),
        prompt=(
            "You are the Bike Repair Co-op's parts clerk. Read "
            "/content/bikecoop/inventory.csv and list every part whose on_hand is "
            "below its reorder_level, with both numbers. Keep the report short. "
            "Do not create or edit any files."
        ),
        tools=["Read", "Glob", "Grep"],
        model=MODEL_NAME,
    ),
    "triage-mechanic": AgentDefinition(
        description=(
            "Ranks the urgent work orders that have a safety issue. Needs no input: "
            "it reads the tickets in /content/bikecoop/work_orders/."
        ),
        prompt=(
            "You are the Bike Repair Co-op's triage mechanic. Read the work orders "
            "in /content/bikecoop/work_orders/. Find every order with "
            "'Priority: urgent' and 'Safety issue: yes'. Rank them from most to "
            "least dangerous to ride, one numbered line each: order ID, skill "
            "needed, and a one-sentence reason. Do not create or edit any files."
        ),
        tools=["Read", "Glob", "Grep"],
        model=MODEL_NAME,
    ),
    "shift-planner": AgentDefinition(
        description=(
            "Assigns volunteers to a given list of work orders. "
            "Needs the order IDs in its prompt."
        ),
        prompt=(
            "You are the Bike Repair Co-op's shift planner. You are given a list "
            "of work order IDs. Read only those orders in "
            "/content/bikecoop/work_orders/ and the volunteer list in "
            "/content/bikecoop/volunteers.txt. Assign each order to one volunteer "
            "whose skills include the skill the order needs, and pick one of that "
            "volunteer's shift days. If no order IDs are given, say so and stop. "
            "Do not create or edit any files."
        ),
        tools=["Read", "Glob", "Grep"],
        model=MODEL_NAME,
    ),
}

for name, agent in AGENTS.items():
    print(f"{name:<16} tools={agent.tools}")

## Cell 7: Helpers for running and inspecting a pipeline

These helpers keep the teaching cells short. None of them orchestrates anything. `run()` sends one prompt, and the others only read the messages afterwards.

| Helper | What it does |
|---|---|
| `run(prompt, extra_tools=[])` | Runs one `query()` with the three specialists registered and `allowed_tools=["Read", "Glob", "Grep", "Agent"] + extra_tools`. Collects every message from the stream and returns the list. |
| `show_result(msgs)` | Prints the final answer from the `ResultMessage`. |
| `timeline(msgs)` | Prints each specialist's start and finish in stream order, then the largest number of specialists that were running at the same moment. |
| `agent_prompts(msgs, look_for=None)` | Prints the prompt the main agent wrote for each Agent call. With `look_for`, it also reports which of those strings appear in each prompt. |
| `subagent_reports(msgs)` | Returns each specialist's final report, as `(name, text)` pairs. |

**Settings inside `run()`.** `"Agent"` in `allowed_tools` auto-approves subagent calls. `Read`, `Glob` and `Grep` let the main agent look at files itself, and `extra_tools` lets a cell add `Write` when the main agent should produce a file. Every call sets `model=MODEL_NAME` and `cwd="/content"`.

The `env` setting `CLAUDE_CODE_DISABLE_BACKGROUND_TASKS` makes the main agent wait for each specialist's report inside the same `query()`. By default, subagents run in the background, and in Colab a single `query()` can end before a background report arrives. With background tasks off, the report always comes back as the Agent tool result.

**How the timeline is built.** A specialist's run shows up in the stream in two ways:

| Signal | Where it appears | Meaning |
|---|---|---|
| `ToolUseBlock` named `Agent` | `AssistantMessage` from the main agent | The main agent asked for a specialist. `block.input["subagent_type"]` names it and `block.id` identifies the call. |
| `TaskStartedMessage` | System message | A task started. Its `tool_use_id` points back to the Agent call. |
| `TaskNotificationMessage` or `TaskUpdatedMessage` with a status in `TERMINAL_TASK_STATUSES` | System message | The task finished. Not every finished task sends a notification, so the helper accepts either. |
| `ToolResultBlock` whose `tool_use_id` equals the call's `id` | `UserMessage` | The specialist's final report came back to the main agent. |

The helper maps every `tool_use_id` to its `subagent_type`, so each line names the specialist. The task messages may not appear in every configuration, so the overlap count at the end uses the Agent call and its result, which are always there.

In [ ]:
import re
from claude_agent_sdk import (
    query, ClaudeAgentOptions, AssistantMessage, UserMessage, ResultMessage,
    ToolUseBlock, ToolResultBlock, TaskStartedMessage, TaskNotificationMessage,
    TaskUpdatedMessage, TERMINAL_TASK_STATUSES,
)

AGENT_TOOL_NAMES = ("Agent", "Task")  # current and older name of the tool


async def run(prompt, extra_tools=[]):
    """Run one query() with the specialists registered; return every message."""
    options = ClaudeAgentOptions(
        agents=AGENTS,
        allowed_tools=["Read", "Glob", "Grep", "Agent"] + extra_tools,
        model=MODEL_NAME,
        cwd="/content",
        env={"CLAUDE_CODE_DISABLE_BACKGROUND_TASKS": "1"},
    )
    msgs = []
    try:
        async for message in query(prompt=prompt, options=options):
            msgs.append(message)
    except Exception as error:
        print(f"Run ended with an error: {error}")
    return msgs


def show_result(msgs):
    """Print the final answer carried by the ResultMessage."""
    for m in msgs:
        if isinstance(m, ResultMessage):
            print(m.result)


def agent_calls(msgs):
    """Map each Agent call's id to the specialist it asked for."""
    calls = {}
    for m in msgs:
        if isinstance(m, AssistantMessage):
            for b in m.content:
                if isinstance(b, ToolUseBlock) and b.name in AGENT_TOOL_NAMES:
                    calls[b.id] = b.input.get("subagent_type", "general-purpose")
    return calls


def timeline(msgs):
    """Print each specialist's start and finish in stream order."""
    calls = agent_calls(msgs)
    owner = {}        # task_id -> specialist name
    finished = set()  # task_ids already reported as finished
    running, most_at_once = set(), 0
    print("TIMELINE (stream order)")
    for m in msgs:
        if isinstance(m, AssistantMessage):
            for b in m.content:
                if isinstance(b, ToolUseBlock) and b.id in calls:
                    running.add(b.id)
                    most_at_once = max(most_at_once, len(running))
                    print(f"  START   {calls[b.id]:<16} Agent call sent")
        elif isinstance(m, TaskStartedMessage) and m.tool_use_id in calls:
            owner[m.task_id] = calls[m.tool_use_id]
            print(f"          {owner[m.task_id]:<16} task started")
        elif isinstance(m, (TaskNotificationMessage, TaskUpdatedMessage)):
            if m.status in TERMINAL_TASK_STATUSES and m.task_id not in finished:
                finished.add(m.task_id)
                name = owner.get(m.task_id) or calls.get(getattr(m, "tool_use_id", None), "?")
                print(f"          {name:<16} task {m.status}")
        elif isinstance(m, UserMessage) and isinstance(m.content, list):
            for b in m.content:
                if isinstance(b, ToolResultBlock) and b.tool_use_id in calls:
                    running.discard(b.tool_use_id)
                    print(f"  FINISH  {calls[b.tool_use_id]:<16} report returned")
    print(f"Most specialists running at once: {most_at_once}")


def agent_prompts(msgs, look_for=None):
    """Print the prompt the main agent wrote for each Agent call."""
    for m in msgs:
        if isinstance(m, AssistantMessage):
            for b in m.content:
                if isinstance(b, ToolUseBlock) and b.name in AGENT_TOOL_NAMES:
                    text = b.input.get("prompt", "")
                    print(f"--- prompt written for {b.input.get('subagent_type')} ---")
                    print(text)
                    if look_for:
                        found = [s for s in look_for if s in text]
                        print(f">>> contains {len(found)} of {len(look_for)}: {', '.join(found)}")
                    print()


def subagent_reports(msgs):
    """Return (specialist name, report text) for every Agent result."""
    calls, reports = agent_calls(msgs), []
    for m in msgs:
        if isinstance(m, UserMessage) and isinstance(m.content, list):
            for b in m.content:
                if isinstance(b, ToolResultBlock) and b.tool_use_id in calls:
                    parts = b.content if isinstance(b.content, list) else [{"text": b.content or ""}]
                    text = "\n".join(p.get("text") or "" for p in parts)
                    reports.append((calls[b.tool_use_id], text))
    return reports


print("Helpers ready: run, show_result, timeline, agent_prompts, subagent_reports")

## Cell 8: Parallel: two independent specialists

The parts check and the triage ranking don't depend on each other. The parts clerk never needs the urgent list, and the triage mechanic never needs the stock levels. Independent jobs are safe to run at the same time, and running them together means both finish in roughly the time of the slower one.

The prompt does three things:

1. Names both specialists, so the main agent delegates instead of doing the work itself.
2. States that the jobs are independent and asks for both to be called together in one step.
3. Asks for one combined summary at the end.

**Parallel is a request, not a switch.** Nothing in `ClaudeAgentOptions` forces parallel execution. The main agent decides, and `timeline()` shows you what it actually did. Two `START` lines before any `FINISH` line, and *Most specialists running at once: 2*, means the specialists overlapped. If they ran one after the other instead, the timeline says so, and the prompt is the place to adjust.

In [ ]:
# LIVE CODE - type during recording

## Cell 9: Sequential: passing results in the prompt

Now the second job depends on the first. The shift planner can only staff the urgent orders once the triage mechanic has found them.

A specialist sees only two things: its own system prompt and the prompt string the main agent writes for it. It never sees the main agent's conversation or another specialist's report. So the five order IDs have to **travel inside the shift planner's prompt**. Your goal prompt says that plainly, and the `shift-planner` description from Cell 6 says the same thing from the other side.

The checks after the run:

- `show_result()` prints the final staffing plan.
- `agent_prompts(seq_msgs, look_for=URGENT_IDS)` prints the prompt the main agent wrote for each specialist and counts how many of the five ground-truth IDs appear in it. For `shift-planner`, you want *contains 5 of 5*.

This cell's messages are reused in Cells 13 and 14, which resume the triage mechanic from this same session.

In [ ]:
# LIVE CODE - type during recording

## Cell 10: Mixed pipeline: the weekly briefing

This cell combines both patterns in one goal prompt:

| Step | Pattern | Who |
|---|---|---|
| 1 | Parallel | `parts-clerk` and `triage-mechanic`, called together |
| 2 | Sequential | `shift-planner`, given the five order IDs from step 1 |
| 3 | Final write | The main agent writes `/content/bikecoop/briefing.md` |

The briefing has exactly three sections: **Low stock**, **Urgent orders** and **Assignments**.

`extra_tools=["Write"]` gives the `Write` tool to the main agent only. The specialists' `tools` lists from Cell 6 don't include it, so they can't create files. The prompt also says *Do not let subagents create files*, which keeps the main agent from asking them to.

The cell deletes any old `briefing.md` first, so the check in the next cell proves this run wrote the file.

In [ ]:
briefing = BASE / "briefing.md"
briefing.unlink(missing_ok=True)

pipeline_prompt = """Prepare this week's Bike Repair Co-op briefing.

Step 1: Use the parts-clerk agent and the triage-mechanic agent. Their jobs are
independent, so call both agents together in one step.
Step 2: When both reports are back, use the shift-planner agent to staff the five
urgent orders from the triage report. Write the five order IDs into its prompt,
because it cannot see the triage report.
Step 3: Write /content/bikecoop/briefing.md yourself, with exactly three sections:
## Low stock, ## Urgent orders, ## Assignments.

Do not let subagents create files. Only you write briefing.md."""

pipe_msgs = await run(pipeline_prompt, extra_tools=["Write"])
show_result(pipe_msgs)
timeline(pipe_msgs)

## Cell 11: Check the briefing against the ground truth

The wording of the briefing changes from run to run, so this cell checks facts rather than sentences:

| Check | Pass means |
|---|---|
| `briefing.md` exists | The pipeline produced its file |
| The three section headings are present | The main agent followed the requested structure |
| Each part in `LOW_STOCK` appears | The parts clerk's findings reached the briefing |
| Each ID in `URGENT_IDS` appears | The triage findings reached the briefing |
| Who called `Write` | Every `Write` call came from the main agent, never from a specialist |

The last check looks at every `Write` tool call in the stream. A call with no `parent_tool_use_id` came from the main agent. A call with a `parent_tool_use_id` would have come from inside a specialist.

In [ ]:
print("briefing.md exists:", briefing.exists())
if briefing.exists():
    text = briefing.read_text()
    print(text)
    print("=" * 60)
    for heading in ["Low stock", "Urgent orders", "Assignments"]:
        print(f"[{'x' if heading.lower() in text.lower() else ' '}] section: {heading}")
    for part in LOW_STOCK:
        print(f"[{'x' if part.lower() in text.lower() else ' '}] low-stock part: {part}")
    for oid in URGENT_IDS:
        print(f"[{'x' if oid in text else ' '}] urgent order: {oid}")

print("\nWrite calls in this run:")
for m in pipe_msgs:
    if isinstance(m, AssistantMessage):
        for b in m.content:
            if isinstance(b, ToolUseBlock) and b.name == "Write":
                who = "main agent" if m.parent_tool_use_id is None else "a SUBAGENT"
                print(f"  {who} wrote {b.input.get('file_path')}")

## Cell 12: Keeping a specialist's output exact

The main agent receives each specialist's final report as the Agent tool result, but it is free to summarise or rephrase that report when it writes its own output. Usually that is what you want. Sometimes you need the specialist's exact words, for example a safety ranking that someone signs off.

The fix lives in the prompt you give the main `query()`: ask for the report to be copied word for word. This cell reruns the Cell 10 pipeline with one extra instruction and writes to `briefing_verbatim.md`.

Then it compares the **Urgent orders** section of both files with the triage mechanic's actual report from each run:

- the two sections side by side,
- and, for each file, how many lines of the triage report appear in it unchanged.

Without the instruction, expect some rewording. With it, the triage report's lines should survive intact.

In [ ]:
import itertools
import textwrap

verbatim = BASE / "briefing_verbatim.md"
verbatim.unlink(missing_ok=True)

verbatim_prompt = (
    pipeline_prompt.replace("briefing.md", "briefing_verbatim.md")
    + "\n\nIn the Urgent orders section, copy the triage-mechanic agent's ranked "
      "list word for word, exactly as it returned it. Do not rephrase, shorten "
      "or reformat it."
)
verb_msgs = await run(verbatim_prompt, extra_tools=["Write"])


def urgent_section(path):
    if not path.exists():
        return f"({path.name} was not written)"
    match = re.search(r"#+\s*Urgent orders(.*?)(?=\n#+\s|\Z)", path.read_text(), re.S | re.I)
    return match.group(1).strip() if match else "(no Urgent orders section)"


def triage_lines(msgs):
    for name, text in subagent_reports(msgs):
        if name == "triage-mechanic":
            return [ln.strip() for ln in text.splitlines()
                    if len(ln.strip()) > 15 and not ln.startswith(("agentId", "<usage>"))]
    return []


def wrap(section):
    return [w for ln in section.splitlines() for w in (textwrap.wrap(ln, 56) or [""])]


left, right = urgent_section(briefing), urgent_section(verbatim)
print(f"{'briefing.md (no instruction)':<58}| briefing_verbatim.md (copy word for word)")
print("-" * 58 + "+" + "-" * 58)
for a, b in itertools.zip_longest(wrap(left), wrap(right), fillvalue=""):
    print(f"{a:<58}| {b}")

for label, path, msgs in [("briefing.md", briefing, pipe_msgs),
                          ("briefing_verbatim.md", verbatim, verb_msgs)]:
    lines, section = triage_lines(msgs), urgent_section(path)
    kept = sum(ln in section for ln in lines)
    print(f"\n{label}: {kept} of {len(lines)} triage report lines kept word for word")

## Cell 13: Capture the session ID and the specialist's agent ID

So far, every question started a fresh specialist. Sometimes you want to go back to the **same** specialist with a follow-up, so it can build on everything it already read and reasoned about. That is resuming.

Resuming needs two IDs from the Cell 9 run:

| ID | Where it comes from | Why you need it |
|---|---|---|
| `session_id` | The `ResultMessage` of the Cell 9 run | Each `query()` starts a new session by default. The specialist's transcript belongs to this session, so you must resume it. |
| `agentId` | A line of the form `agentId: <id>` in the `triage-mechanic` report text | It tells the main agent which specialist to continue. |

The `agentId` line is added by the agent runtime to the Agent tool result, not written by the specialist. Custom agents such as `triage-mechanic` include it. The cell parses it with a regular expression and prints the end of the raw report so you can see the line's exact format.

In [ ]:
session_id = next((m.session_id for m in seq_msgs if isinstance(m, ResultMessage)), None)

triage_id, triage_raw = None, ""
for name, text in subagent_reports(seq_msgs):
    if name == "triage-mechanic":
        triage_raw = text
        if match := re.search(r"agentId:\s*([\w-]+)", text):
            triage_id = match.group(1)

print("session_id:              ", session_id)
print("triage-mechanic agentId: ", triage_id)
print("\nLast lines of the triage-mechanic report as the main agent received it:")
print("\n".join(triage_raw.splitlines()[-6:]))
if not triage_id:
    print("\nNo agentId was found in the triage-mechanic result, so there is "
          "no specialist to resume. Rerun Cell 9, then this cell.")

## Cell 14: Resume the triage mechanic

This is a second `query()` that continues the Cell 9 session. It is not orchestration: it simply picks up the same conversation.

| Option | Value | Why |
|---|---|---|
| `resume` | `session_id` from Cell 13 | Continue the session that holds the triage mechanic's transcript |
| `agents` | The same `AGENTS` dict | A custom specialist must be defined in both queries |
| `allowed_tools` | Same as `run()` | The main agent can call the Agent tool again |
| `model`, `cwd`, `env` | Same as `run()` | Same session settings as the original run |

The prompt names the agent ID and asks a follow-up that only makes sense with the earlier ranking in mind. A resumed specialist keeps its full history: the files it read, its tool results and its reasoning.

After the run, the cell shows how the main agent reached the specialist:

- every tool call the main agent made, with its input,
- any permission denials from the `ResultMessage`,
- whether the same `agentId` appears in the new result.

If the main agent answers without calling any tool, it used its own memory of the triage report rather than the specialist, and the tool-call list will be empty.

In [ ]:
resume_msgs = []
if session_id and triage_id:
    followup = (
        f"Resume agent {triage_id}. Of the five urgent orders you ranked, "
        "which one could safely wait until next week, and why?"
    )
    try:
        async for message in query(
            prompt=followup,
            options=ClaudeAgentOptions(
                agents=AGENTS,
                allowed_tools=["Read", "Glob", "Grep", "Agent"],
                model=MODEL_NAME,
                cwd="/content",
                env={"CLAUDE_CODE_DISABLE_BACKGROUND_TASKS": "1"},
                resume=session_id,
            ),
        ):
            resume_msgs.append(message)
    except Exception as error:
        print(f"Run ended with an error: {error}")

    show_result(resume_msgs)

    print("\nTool calls made by the main agent:")
    for m in resume_msgs:
        if isinstance(m, AssistantMessage) and m.parent_tool_use_id is None:
            for b in m.content:
                if isinstance(b, ToolUseBlock):
                    print(f"  {b.name}: {str(b.input)[:200]}")

    for m in resume_msgs:
        if isinstance(m, ResultMessage):
            print("Permission denials:", m.permission_denials or "none")

    new_ids = [re.search(r"agentId:\s*([\w-]+)", t) for _, t in subagent_reports(resume_msgs)]
    new_ids = [m.group(1) for m in new_ids if m]
    print("agentId in the new result:", new_ids or "none")
    print("Same specialist as Cell 9:", triage_id in new_ids)
else:
    print("Nothing to resume. Run Cells 9 and 13 first.")